In [1]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Flatten,MaxPooling2D,Input,BatchNormalization
from tensorflow.keras.models import Sequential, Model
from tensorflow.python.framework import ops
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
import cv2
import dotenv
from torch.utils.data import DataLoader, TensorDataset
import torch

In [2]:
# get environment variables
dotenv.load_dotenv()

True

In [3]:
dir = os.getenv("data_path") + '\\satellite\\train'
x = []
y = []
for direct in os.listdir(dir):
    print("Loading dataset training {}".format(direct))
    for filename in os.listdir(os.path.join(dir,direct)):
        img_path = os.path.join(dir,direct,filename)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (32,32))
        img = np.array(img)
        img = img/255
        x.append(img)
        y.append(direct)

Loading dataset training fire
Loading dataset training nofire


In [4]:
dir = os.getenv("data_path") + '\\satellite\\val'
x_val=[]
y_val=[]
for direct in os.listdir(dir):
    print("Loading dataset validation {}".format(direct))
    for filename in os.listdir(os.path.join(dir,direct)):
        img_path = os.path.join(dir,direct,filename)
        image = cv2.imread(img_path)
        if image is None:
            print("Error loading image: {}".format(img_path))
            continue
        image = cv2.resize(image,(32,32))
        image = np.array(image)
        image = image/255
        x_val.append(image)
        y_val.append(direct)

Loading dataset validation fire
Error loading image: C:\Users\panje\Documents\GitHub\DS-4420-Final-Project\data\satellite\val\fire\desktop.ini
Loading dataset validation nofire


In [5]:
# Encode labels and one-hot encode them for training data
label_encode = LabelEncoder()
label_y = label_encode.fit_transform(y)  # Encode training labels as integers
one_hot_y = to_categorical(label_y)  # One-hot encode the training labels

# Encode labels and one-hot encode them for testing data
label_y_val = label_encode.transform(y_val)  # Encode testing labels using the same encoder
one_hot_y_val = to_categorical(label_y_val)  # One-hot encode the testing labels

# Convert features to NumPy arrays
x = np.array(x)  # Training features
x_val = np.array(x_val)  # Testing features

# Convert data to PyTorch tensors
x_train_tensor = torch.tensor(x, dtype=torch.float32)
y_train_tensor = torch.tensor(one_hot_y, dtype=torch.float32)

x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
y_val_tensor = torch.tensor(one_hot_y_val, dtype=torch.float32)

# Create TensorDatasets
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Check shapes
print("x_train_tensor shape:", x_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("x_val_tensor shape:", x_val_tensor.shape)
print("y_val_tensor shape:", y_val_tensor.shape)

x_train_tensor shape: torch.Size([1887, 32, 32, 3])
y_train_tensor shape: torch.Size([1887, 2])
x_val_tensor shape: torch.Size([402, 32, 32, 3])
y_val_tensor shape: torch.Size([402, 2])


In [6]:
model = Sequential()

# Add a 2D convolutional layer with 32 filters, kernel size of (3, 3), 'same' padding, ReLU activation, and input shape (32, 32, 3)
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))

# Add another 2D convolutional layer with 64 filters, kernel size of (3, 3), 'same' padding, and ReLU activation
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))

# Add a max pooling layer with a pool size of (2, 2)
model.add(MaxPooling2D(pool_size=(2, 2)))

# Add a batch normalization layer to normalize the activations
model.add(BatchNormalization())

# Flatten the 2D feature maps into a 1D feature vector
model.add(Flatten())

# Add a dropout layer with a dropout rate of 0.2 to prevent overfitting
model.add(Dropout(0.2))

# Add a dense (fully connected) layer with 64 units and ReLU activation
model.add(Dense(64, activation='relu'))

# Add the output dense layer with 2 units (for binary classification) and sigmoid activation
# model.add(Dense(2, activation='softmax'))
model.add(Dense(2, activation='sigmoid'))

c:\Users\panje\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,048,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,068,418 (4.08 MB)

 Trainable params: 1,068,290 (4.08 MB)

 Non-trainable params: 128 (512.00 B)

In [8]:
model.compile(optimizer="SGD", loss="categorical_crossentropy", metrics=["accuracy"])

# Train the model using the training data
history = model.fit(train_loader, epochs=50, validation_data=val_loader, verbose =1)

# Evaluate the model using the validation data
loss, accuracy = model.evaluate(val_loader)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

Epoch 1/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6857 - loss: 0.6791 - val_accuracy: 0.7338 - val_loss: 0.6647
Epoch 2/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7765 - loss: 0.4555 - val_accuracy: 0.7264 - val_loss: 0.6192
Epoch 3/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8444 - loss: 0.3472 - val_accuracy: 0.7139 - val_loss: 0.5936
Epoch 4/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8399 - loss: 0.3385 - val_accuracy: 0.8010 - val_loss: 0.5603
Epoch 5/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8755 - loss: 0.3042 - val_accuracy: 0.7264 - val_loss: 0.5533
Epoch 6/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8911 - loss: 0.2601 - val_accuracy: 0.8035 - val_loss: 0.4768
Epoch 7/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9039 - loss: 0.2244 - val_accuracy: 0.8085 - val_loss: 0.4451
Epoch 8/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9467 - loss: 0.1752 - val_accuracy: 0.7338 - val_loss